## 03．データ分割

１．計算した分子記述子と温度・湿度などの環境変数・配合情報を合体

２．WVPの値を対数変換（log1p）

３．RDKitの欠損値を完全に排除した上で、訓練用（80%）と検証用（20%）に分割

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\01_cleaned_raw.csv")
df_desc = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\02_molecular_descriptors.csv")

def prepare_dataset(df_clean, df_desc):
    print("--- Step 3: 特徴量結合・対数変換・データ分割 ---")

    # 環境変数・配合比率と分子記述子を結合
    X_raw = pd.concat([df_clean[['humidity', 'temperature', 'material_concentration', 'proportion']].reset_index(drop=True), df_desc], axis=1)
    y_raw = df_clean['wvp'].reset_index(drop=True)

    # 欠損値（SMILESエラー行など）の最終削除
    valid_idx = X_raw.notna().all(axis=1)
    X = X_raw[valid_idx].copy()

    # wvpを対数変換
    y = np.log1p(y_raw[valid_idx])

    # 訓練用とテスト用に分割
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

#実行用
if __name__ == "__main__":
    # 1. 関数を実行して、データを分割し結果を受け取る
    X_train, X_test, y_train, y_test = prepare_dataset(df_clean, df_desc)

    # 2. 画面に確認用のログを出力する
    print(f"訓練データ（特徴量）の形状: {X_train.shape}")
    print(f"テストデータ（特徴量）の形状: {X_test.shape}")

    export_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\03_split_dataset.npz"

    np.savez(export_path,
            X_train=X_train.values,
            X_test=X_test.values,
            y_train=y_train.values,
            y_test=y_test.values,
            columns=X_train.columns.tolist())

    print("Step 3 の分割データを正常に保存しました！")

--- Step 3: 特徴量結合・対数変換・データ分割 ---
訓練データ（特徴量）の形状: (4012, 25)
テストデータ（特徴量）の形状: (1003, 25)
Step 3 の分割データを正常に保存しました！
